# 標準獲得関数の比較：単目的ベイズ最適化の実践ガイド

同じ `SingleTaskGP` の事後分布から、**LogEI・LogPI・UCB・qLogEI・qUCB・qSimpleRegret・qKnowledgeGradient** がどの候補を選ぶか比較します。

学習 → 事後予測 → 獲得関数最大化 → 候補評価 → データ追加 → 再学習を通して、**探索（exploration）と活用（exploitation）** の違いを理解します。

標準獲得関数はBoTorchの既存公開APIを直接使い、Notebook専用の獲得関数クラスは定義しません。

関連理論: [Improvement](../../docs/theory/acquisition/02_improvement.md) / [Confidence Bound](../../docs/theory/acquisition/03_confidence_bound.md) / [Lookahead](../../docs/theory/acquisition/06_lookahead.md) / [Selection Guide](../../docs/theory/acquisition/12_selection_guide.md)


## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.analytic import (
    LogExpectedImprovement,
    LogProbabilityOfImprovement,
    UpperConfidenceBound,
)
from botorch.acquisition.knowledge_gradient import qKnowledgeGradient
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.monte_carlo import qSimpleRegret, qUpperConfidenceBound
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### このNotebookの問題設定

入力 `x` は0〜1の1変数、目的関数 `f(x)` は実験で測る品質スコアだと考えます。**最大化問題**なので、値が大きいほど望ましい条件です。最初に6点だけ観測し、未観測領域の値はGPの事後分布で推定します。

`model.fit()` は観測済みデータに基づきGPのパラメータを推定します。獲得関数の最大化は**次に評価する入力条件**を選ぶ処理であり、GP学習とは別です。

## 2. Problem and surrogate

最大化問題として統一します。noise-freeの簡潔な比較なのでimprovement familyの基準は観測`train_Y.max()`を使います。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.35 * torch.cos(5 * torch.pi * X)


train_X = torch.tensor([[0.05], [0.18], [0.38], [0.58], [0.78], [0.95]])
train_Y = objective(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()
print("best observed value:", train_Y.max())

## 3. Same posterior, different utilities

acquisition familyを変えてもsurrogate posteriorは同じです。違うのはposteriorをdecision utilityへ変換する規則です。

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), mean, label="posterior mean")
ax.fill_between(grid.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.legend()
plt.show()

## 4. Analytic q=1: LogEI, LogPI, UCB

- LogEI: improvement量の期待値をlog-domainで安定に扱います。
- LogPI: improvement eventの確率を重視します。
- UCB: posterior meanとuncertainty bonusを直接組み合わせます。

Log formulationは別の探索原理ではありません。

### EI・PI・UCBを使い分ける考え方

- **LogEI**：現在の最良値をどれだけ上回りそうか（改善量）を重視します。Log値が負でも「改善が負」という意味ではありません。
- **LogPI**：改善が起こる確率を重視します。改善幅が小さくても確率が高ければ選び得ます。
- **UCB**：`posterior mean + uncertainty bonus` に相当し、`beta` を大きくすると不確実性をより重視します。

3手法は同じGPを参照しますが、目的とする意思決定が異なります。獲得関数値の単位・スケールも異なるため、**値の大きさで手法間の優劣を比較しません**。

In [ ]:
analytic_acqfs = {
    "LogEI": LogExpectedImprovement(model=model, best_f=train_Y.max()),
    "LogPI": LogProbabilityOfImprovement(model=model, best_f=train_Y.max()),
    "UCB": UpperConfidenceBound(model=model, beta=0.2),
}

analytic_candidates = {}
for name, acqf in analytic_acqfs.items():
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    analytic_candidates[name] = candidate
    print(name, "candidate=", candidate.item(), "value=", value.item())

### 獲得関数の形を直接見てみる

1次元なので候補全体のスコアを曲線で表示できます。各曲線の最大位置が、最適化器が探す候補の目安になります。比較しやすいよう**各曲線を独立に標準化**して表示します（最適化そのものには標準化値を使いません）。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, acqf in analytic_acqfs.items():
    with torch.no_grad():
        values = acqf(grid.unsqueeze(-2)).reshape(-1)
    scaled = (values - values.min()) / (values.max() - values.min()).clamp_min(1e-12)
    ax.plot(grid.squeeze(-1), scaled, label=name)
    ax.axvline(analytic_candidates[name].item(), linestyle="--", alpha=0.35)
ax.set(xlabel="x", ylabel="Individually normalized acquisition value", title="Different criteria select different locations")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 5. MC q=1: qLogEI, qUCB, qSimpleRegret

q=1でもMC formulationを使えます。`qSimpleRegret`はposterior utilityとして候補値そのものを重視し、improvement thresholdを必要としません。

In [ ]:
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([128]), seed=123)
mc_acqfs = {
    "qLogEI": qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
        sampler=sampler,
    ),
    "qUCB": qUpperConfidenceBound(model=model, beta=0.2, sampler=sampler),
    "qSimpleRegret": qSimpleRegret(model=model, sampler=sampler),
}

mc_candidates = {}
for name, acqf in mc_acqfs.items():
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    mc_candidates[name] = candidate
    print(name, "candidate=", candidate.item(), "value=", value.item())

## 6. Candidate locations are the decision output

acquisition valueの絶対値をfamily間で比較して優劣を付けません。utilityの定義・scaleが違うためです。比較対象は各criterionがどの意思決定を表すかと、結果としてどこをcandidateにするかです。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), objective(grid).squeeze(-1), label="objective")
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")

all_candidates = {**analytic_candidates, **mc_candidates}
for name, candidate in all_candidates.items():
    y = objective(candidate).item()
    ax.scatter(candidate.item(), y, marker="x", s=70, label=name)

ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.legend(ncol=2)
plt.show()

## 7. Knowledge Gradient is structurally different

KGはcandidate自身のimprovementではなく、観測後のterminal decision valueを評価します。`qKnowledgeGradient`はone-shot acquisitionなので、optimizer内部ではreal candidateに加えてauxiliary fantasy decision rowsも最適化します。

In [ ]:
kg = qKnowledgeGradient(
    model=model,
    num_fantasies=4,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([4]), seed=321),
)
augmented_q = kg.get_augmented_q_batch_size(q=1)
print("real q:", 1)
print("augmented q:", augmented_q)

## 8. qKG candidate generation and extraction

`optimize_acqf`はone-shot acquisitionの`extract_candidates` semanticsを認識し、返却値は実際に評価するcandidateだけになります。auxiliary rowsを実験点として扱いません。

In [ ]:
kg_candidate, kg_value = optimize_acqf(
    acq_function=kg,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)

assert kg_candidate.shape == torch.Size([1, 1])
print("qKG actual candidate:", kg_candidate)
print("qKG value:", kg_value)

## 9. One candidate evaluation

ここでは代表としてqLogEI candidateを評価します。acquisition比較のNotebookなので、各手法を順番に評価してtraining dataを変えることはしません。

### 1回の実験後に何が変わるか

以下では、qLogEIが選んだ1点を実験で評価し、観測データへ追加します。その後、**同じモデル構造で再学習**して、事後平均と不確実性がどう変化するか比較します。

人工目的関数は説明のための実験装置の代役です。実務では `objective(candidate)` を実際の実験・シミュレーション結果に置き換えます。

In [ ]:
candidate = mc_candidates["qLogEI"]
candidate_Y = objective(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)
print("candidate:", candidate)
print("evaluation:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

### 更新後のGPと次の候補

追加観測の前後で、平均予測・±2標準偏差・獲得関数の提案が変わる様子を確認します。獲得関数は**更新後のモデルと更新後の最良観測値**を使って再作成する必要があります。

In [ ]:
updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()
with torch.no_grad():
    updated_posterior = updated_model.posterior(grid)
updated_mean = updated_posterior.mean.squeeze(-1)
updated_std = updated_posterior.variance.clamp_min(0).sqrt().squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, center, uncertainty, title in (
    (axes[0], mean, std, "Before next observation"),
    (axes[1], updated_mean, updated_std, "After observation and fit"),
):
    ax.plot(grid.squeeze(-1), objective(grid).squeeze(-1), label="true objective")
    ax.plot(grid.squeeze(-1), center.detach(), label="posterior mean")
    ax.fill_between(grid.squeeze(-1), (center - 2 * uncertainty).detach(), (center + 2 * uncertainty).detach(), alpha=0.2)
    ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="initial data")
    ax.scatter(candidate.squeeze(-1), candidate_Y.squeeze(-1), marker="x", s=80, label="new observation")
    ax.set(xlabel="x", title=title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

next_acqf = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([128]), seed=124),
)
next_candidate, next_value = optimize_acqf(
    acq_function=next_acqf, bounds=bounds, q=1, num_restarts=5, raw_samples=64
)
print("再学習後の次の提案:", next_candidate, "acquisition:", next_value)
assert updated_X.shape[0] == train_X.shape[0] + 1
assert next_candidate.shape == (1, 1)

## 10. What changes for q > 1

`qLogEI`や`qUCB`の`q>1`はpointwise score上位q点ではありません。joint posterior samplesを使ってcandidate set全体を価値付けします。

```text
q=1 analytic
    marginal posterior momentsで閉形式を使える場合がある

q>1 MC
    joint posterior / candidate dependenceが重要

qKG
    q real candidates + auxiliary fantasy decision rows
```

Batch / pending / noisy semanticsは専用Phaseで扱います。

## 10.1 q=2を実際に最適化する

`qLogExpectedImprovement` は `q=2` にも対応します。2点の獲得関数値は、**各点の単独スコアを足すだけではなく、同時事後分布を使って評価**されます。候補が近すぎると情報が重複する場合があるため、単一点の上位2点を選ぶ方法とは異なります。

ここでは最初のモデルを使い、2点を一括提案する例を示します。

In [ ]:
batch_acqf = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([128]), seed=125),
)
batch_candidates, batch_value = optimize_acqf(
    acq_function=batch_acqf, bounds=bounds, q=2,
    num_restarts=5, raw_samples=64,
)
print("同時提案する2点:", batch_candidates.flatten().tolist())
print("joint qLogEI:", batch_value.item())
assert batch_candidates.shape == (2, 1)

## 11. Selection semantics

| Family | Main decision quantity | Typical question |
| --- | --- | --- |
| LogEI / qLogEI | expected improvement | incumbentをどれだけ改善できるか |
| LogPI | probability of improvement | 改善eventが起きる確率は高いか |
| UCB / qUCB | optimistic posterior utility | meanとuncertaintyをどうtrade-offするか |
| qSimpleRegret | posterior utility of selected points | 良い最終点を選ぶにはどこか |
| qKG | value of information | 観測後のterminal decisionがどれだけ改善するか |

acquisition valueそのものを異なるfamily間で横並び比較しません。

## 12. Important boundaries

- noiseが強い場合は単純なobserved `best_f`よりqLogNEI等を検討します。
- q-batchとsequential top-q rankingを同一視しません。
- `X_pending`とKGのfantasy rowsは別物です。
- outcome constraintとcandidate/input-space constraintは別レイヤーです。
- multi-objective、multi-fidelity、active learningは別のdecision purposeです。
- standard acquisitionはBoTorch native pathを優先し、robotorchan wrapperを増やしません。

## 13. References

数式、posterior requirement、q/noisy/lookaheadの詳細は冒頭のTheory章を参照してください。APIの現在の推奨は`docs/optimization/standard_acquisition.md`にあります。

## 14. 公開APIと使い分けのまとめ

| 処理 | 公開API | このNotebookでの役割 |
|---|---|---|
| GPの作成・学習 | `robotorchan.models.SingleTaskGP` / `model.fit()` | 観測から事後分布を推定 |
| 解析的な単一点獲得関数 | `LogExpectedImprovement` / `LogProbabilityOfImprovement` / `UpperConfidenceBound` | 改善・改善確率・楽観的評価 |
| MC獲得関数 | `qLogExpectedImprovement` / `qUpperConfidenceBound` / `qSimpleRegret` | MCによる候補価値評価 |
| Lookahead | `qKnowledgeGradient` | 観測後の意思決定価値 |
| 候補の最適化 | `botorch.optim.optimize_acqf` | 獲得関数の最大化 |
| 観測・再学習 | `objective(candidate)` → データ追加 → `fit()` | BOループの1ステップ |

**実務上の注意：** 候補を最大化する獲得関数と、実験で最大化したい目的関数は異なります。獲得関数は次にどこを測るかを決め、目的関数は測定して初めて値が分かります。